# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

To explore the record sets and their structure, we'll extract their `@id`s and info using the metadata object. All references to entities are strictly using their `@id`.

In [ ]:
# List the record sets and fields (using @id references)
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in the dataset. Please verify the schema or contact the dataset maintainer.")
else:
    print("Record Sets:")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
        print("  Fields in this record set:")
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"    * Field @id: {field['@id']} | Name: {field.get('name', 'N/A')} | DataType: {field.get('dataType', 'N/A')}")
        else:
            print("    No fields found.")
        print("  Columns in this record set:")
        if 'columns' in rs:
            for col in rs['columns']:
                print(f"    * Column @id: {col['@id']} | Name: {col.get('name', 'N/A')} | Source: {col.get('source', 'N/A')}")
        else:
            print("    No columns found.")
        print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
**All references are by `@id`, not by names.**
Below, we dynamically extract the available record set `@id`s and demonstrate how to load each as a pandas DataFrame using `mlcroissant`.

In [ ]:
# This cell will extract the records from each available record set by @id.
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets] if dataset.metadata.record_sets else []

dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded data for RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

# For demo, select the first record set
if record_sets_ids:
    primary_record_set_id = record_sets_ids[0]
    print(f"Using record set @id: {primary_record_set_id} for additional analysis.")
else:
    primary_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations may include removing outliers, transforming distributions, or grouping by key attributes.

Below, we demonstrate EDA for one numeric field in the first record set. **All fields are referenced by `@id`.**

In [ ]:
# Select a numeric field for analysis using its @id
if primary_record_set_id:
    df = dataframes[primary_record_set_id]
    # Attempt to dynamically select a numeric field
    # Use the metadata record set's fields to help
    primary_rs_meta = next((rs for rs in dataset.metadata.record_sets if rs['@id']==primary_record_set_id), {})
    numeric_field_id = None
    group_field_id = None

    # Examine fields for numeric types
    if 'fields' in primary_rs_meta:
        for field in primary_rs_meta['fields']:
            if 'dataType' in field:
                if field['dataType'] in ('schema:Float', 'schema:Integer', 'schema:Number'):
                    numeric_field_id = field['@id']
                    break
        # Select a categorical field for grouping if available
        for field in primary_rs_meta['fields']:
            if 'dataType' in field and field['dataType'] == 'schema:Text':
                group_field_id = field['@id']
                break

    if numeric_field_id and numeric_field_id in df.columns:
        # Filtering
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA in the first record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we produce a histogram for a numeric field referenced by its `@id` and, if possible, a boxplot grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Histogram of Numeric Field ({numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion

- In this notebook, we explored the dataset using the FAIR^2 Croissant schema and mlcroissant library.
- Record sets, fields, and columns were referenced strictly by their `@id` attributes for reproducibility.
- Data analysis included filtering, normalization, and grouping on numeric fields; visualizations demonstrated distributions and group differences.
- For further analysis or modeling, continue using entity `@id`s to ensure consistency with the Croissant metadata.

For more information or to report issues with the dataset, please consult [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) or the FAIR^2 documentation.